# 690 Retrieval Optimization

690개 문서 기반 Chroma DB에서 retrieval 성능을 다시 튜닝한 기록입니다.

기준 실험은 80번이고, 125개 DB에서 효과가 있었던 query decomposition, RRF, document-level selection 계열을 690 DB에 다시 적용했습니다.

최종적으로 가장 좋았던 조합은 91번입니다.

- vector store: Chroma
- embedding model: `nlpai-lab/KURE-v1`
- retriever: dense
- query decomposition: conditional + RRF
- per-query candidates: 75
- final selection: document scoring, `mean_top_n`, top 3 chunks 평균
- document score candidates: 300


## 1. 결과 로그 불러오기

`eval/evaluation/outputs/eval/experiment_logs/phase1_retrieval_experiments.csv`에 누적된 평가 로그에서 690 실험만 가져옵니다.

In [ ]:
from pathlib import Path
import csv
import json
import pandas as pd

LOG_PATH = Path('../eval/evaluation/outputs/eval/experiment_logs/phase1_retrieval_experiments.csv')

EXPERIMENT_LABELS = {
    '80_dense_qdecomp_rrf_diverse250_kure_chroma_690_canonical': '80 baseline: per50 + diverse250',
    '81_dense_qdecomp_rrf_diverse250_keyword_rerank_kure_chroma_690_canonical': '81 keyword rerank',
    '82_dense_qdecomp_rrf_per50_diverse200_kure_chroma_690_canonical': '82 per50 + diverse200',
    '83_dense_qdecomp_rrf_per50_diverse300_kure_chroma_690_canonical': '83 per50 + diverse300',
    '84_dense_qdecomp_rrf_per50_diverse400_kure_chroma_690_canonical': '84 per50 + diverse400',
    '85_dense_qdecomp_rrf_per50_diverse250_source_file_kure_chroma_690_canonical': '85 source_file diversity',
    '86_dense_qdecomp_rrf_per75_diverse250_kure_chroma_690_canonical': '86 per75 + diverse250',
    '87_dense_qdecomp_rrf_per75_diverse300_kure_chroma_690_canonical': '87 per75 + diverse300',
    '88_dense_qdecomp_rrf_per75_diverse400_kure_chroma_690_canonical': '88 per75 + diverse400',
    '89_dense_qdecomp_rrf_per100_diverse300_kure_chroma_690_canonical': '89 per100 + diverse300',
    '90_dense_qdecomp_rrf_per100_diverse400_kure_chroma_690_canonical': '90 per100 + diverse400',
    '91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical': '91 docscore mean3, candidates300',
    '92_dense_qdecomp_rrf_per75_docscore_max_300_kure_chroma_690_canonical': '92 docscore max, candidates300',
    '93_dense_qdecomp_rrf_per75_docscore_sum3_300_kure_chroma_690_canonical': '93 docscore sum3, candidates300',
}

def load_latest_experiments(log_path: Path) -> pd.DataFrame:
    latest = {}
    with log_path.open(encoding='utf-8-sig') as file:
        for row in csv.DictReader(file):
            name = row['experiment_name']
            if name not in EXPERIMENT_LABELS:
                continue
            by_type = json.loads(row.get('by_type_summary_json') or '[]')
            by_diff = json.loads(row.get('by_difficulty_summary_json') or '[]')
            type_b = next((item for item in by_type if item.get('type') == 'B'), {})
            type_e = next((item for item in by_type if item.get('type') == 'E'), {})
            hard = next((item for item in by_diff if item.get('difficulty') == '상'), {})
            latest[name] = {
                'experiment_name': name,
                'setting': EXPERIMENT_LABELS[name],
                'hit_at_5': float(row['overall_hit_at_5']),
                'mrr_at_5': float(row['overall_mrr_at_5']),
                'ndcg_at_5': float(row['overall_ndcg_at_5']),
                'type_b_ndcg': type_b.get('ndcg_at_5'),
                'type_b_doc_recall': type_b.get('doc_recall_at_5'),
                'type_b_multi_doc_recall': type_b.get('multi_doc_recall_at_5'),
                'type_e_ndcg': type_e.get('ndcg_at_5'),
                'type_e_doc_recall': type_e.get('doc_recall_at_5'),
                'hard_ndcg': hard.get('ndcg_at_5'),
                'run_datetime': row.get('run_datetime'),
                'predictions_path': row.get('predictions_path'),
            }
    return pd.DataFrame(latest.values()).sort_values('ndcg_at_5', ascending=False).reset_index(drop=True)

summary = load_latest_experiments(LOG_PATH)
summary


## 2. nDCG 기준 순위

전체 nDCG@5 기준으로 보면 document scoring 방식이 가장 좋았습니다. 특히 91번은 80번 기준 실험보다 MRR과 nDCG가 모두 올랐습니다.

In [ ]:
cols = [
    'setting',
    'hit_at_5',
    'mrr_at_5',
    'ndcg_at_5',
    'type_b_ndcg',
    'type_b_doc_recall',
    'type_e_ndcg',
    'hard_ndcg',
]

ranked = summary[cols].copy()
ranked.index = ranked.index + 1
ranked


## 3. 기준 실험 대비 개선폭

80번을 기준으로 가장 좋은 91번이 얼마나 개선됐는지 계산합니다.

In [ ]:
baseline = summary[summary['experiment_name'] == '80_dense_qdecomp_rrf_diverse250_kure_chroma_690_canonical'].iloc[0]
best = summary.iloc[0]

improvement = pd.DataFrame([
    {'metric': 'MRR@5', 'baseline_80': baseline['mrr_at_5'], 'best': best['mrr_at_5'], 'delta': best['mrr_at_5'] - baseline['mrr_at_5']},
    {'metric': 'nDCG@5', 'baseline_80': baseline['ndcg_at_5'], 'best': best['ndcg_at_5'], 'delta': best['ndcg_at_5'] - baseline['ndcg_at_5']},
    {'metric': 'Type B nDCG@5', 'baseline_80': baseline['type_b_ndcg'], 'best': best['type_b_ndcg'], 'delta': best['type_b_ndcg'] - baseline['type_b_ndcg']},
    {'metric': 'Type E nDCG@5', 'baseline_80': baseline['type_e_ndcg'], 'best': best['type_e_ndcg'], 'delta': best['type_e_ndcg'] - baseline['type_e_ndcg']},
])
improvement


## 4. 시각화

상위 10개 조합의 nDCG@5, MRR@5를 비교합니다. `matplotlib`이 없으면 아래 셀에서 안내 메시지만 출력됩니다.

In [ ]:
try:
    import matplotlib.pyplot as plt

    plot_df = summary.head(10).set_index('setting')[['ndcg_at_5', 'mrr_at_5']]
    ax = plot_df.iloc[::-1].plot(kind='barh', figsize=(10, 6), width=0.75)
    ax.set_xlim(0.84, 0.96)
    ax.set_title('Top 10 Retrieval Settings on 690 DB')
    ax.set_xlabel('score')
    ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print('matplotlib이 설치되어 있지 않습니다. 필요하면 `pip install matplotlib` 후 다시 실행하세요.')


## 5. 해석

- 80번 기준 실험은 이미 꽤 높은 성능이었지만, 690 DB에서는 125 DB보다 문서 수와 유사 문서가 많아서 type B, type E 쪽 점수가 내려갔습니다.
- diversity 후보를 200에서 300 이상으로 늘리면 성능이 조금 좋아졌지만, 300과 400은 거의 같았습니다. 후보 폭은 300 근처에서 충분해 보입니다.
- per-query candidates는 50보다 75가 조금 좋았고, 100은 오히려 MRR이 떨어졌습니다. 후보를 너무 많이 늘리면 노이즈가 같이 들어오는 것으로 보입니다.
- keyword rerank는 690에서도 좋지 않았습니다. 특히 type B의 여러 정답 문서 recall을 크게 떨어뜨렸습니다.
- 가장 좋았던 91번은 document scoring입니다. 한 문서의 여러 chunk 점수를 평균내서 문서 단위로 랭킹한 방식이고, type E nDCG가 크게 개선됐습니다.

현재 최종 선택은 91번으로 보는 것이 좋습니다.

## 6. 최종 세팅 재현 명령어

아래 명령어를 실행하면 91번 세팅을 다시 생성하고 평가할 수 있습니다.

In [ ]:
best_generate_cmd = '''.venv/bin/python scripts/generate_eval_predictions.py \\
  --canonical-only \\
  --retriever dense \\
  --chunks data/processed/chunks_v2_690.jsonl \\
  --vector-store chroma \\
  --index-dir indexes/chroma_kure_v1_chunks_v2_690 \\
  --embedding-preset kure \\
  --embedding-provider huggingface \\
  --query-decomposition \\
  --decomposition-candidates-per-query 75 \\
  --decomposition-selection rrf \\
  --decomposition-conditional \\
  --decomposition-min-subqueries 2 \\
  --document-scoring \\
  --doc-score-candidates 300 \\
  --doc-score-method mean_top_n \\
  --doc-score-top-n 3 \\
  --doc-score-key doc_id \\
  --context-max-chars 1200 \\
  --output outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl \\
  --progress-every 100'''

best_eval_cmd = '''.venv/bin/python eval/scripts/run_evaluation.py \\
  --predictions outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl \\
  --canonical-only \\
  --include-analysis-metrics \\
  --experiment-name 91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical'''

print(best_generate_cmd)
print('\n' + best_eval_cmd)


## 7. 추가 실험 후보

지금 단계에서는 retrieval만 더 미세하게 튜닝하는 것보다, generation으로 넘겼을 때 실제 답변 품질을 보는 쪽이 더 중요합니다. 그래도 retrieval에서 더 해본다면 아래 정도가 후보입니다.

- 91번 세팅을 기준으로 `doc-score-candidates`를 200, 400으로 비교
- `doc-score-top-n`을 2, 4로 비교
- type B만 따로 보고 query decomposition subquery 생성 규칙 보완
- 데이터 쪽에서 중복/유사 공고의 canonical doc id 정리
